# 🔍 LABORATORIO N° 13 — Fundamentos de Gestión de Datos
## Semana 13: Calidad de Datos
### Lab D13: Análisis de Calidad del Dataset Final del Proyecto

---
| | |
|---|---|
| **Curso** | Fundamentos de Gestión de Datos |
| **Docente** | Pilar Rocío Sayán Mejía |
| **Periodo** | 2026-I — UNIDAD 4 (Semanas 13 a 16) |
| **Semana** | 13 |

---
**Nombre del estudiante / equipo:** `_______________________________`

**Sección:** `______`  &nbsp;&nbsp;&nbsp;  **Fecha:** `______`

## 📌 Elemento de la Capacidad Terminal

> Genera un reporte de calidad automatizado de la base de datos del proyecto identificando problemas en las dimensiones de completitud, exactitud, consistencia, unicidad y oportunidad, e implementa correcciones automáticas en el pipeline de datos con Python y pandas.

---
## 📚 ACTIVIDAD 1: Revisión de Conceptos — Dimensiones de Calidad de Datos

Complete con definiciones propias. Use ejemplos de su proyecto.

| # | Concepto / Principio | Definición con sus propias palabras |
|---|---|---|
| 1 | ¿Qué es la calidad de datos y por qué es crítica en proyectos reales? | *(complete aquí)* |
| 2 | Dimensión de **completitud**: ¿qué mide? ¿cómo se detecta y corrige en Python? | *(complete aquí)* |
| 3 | Dimensión de **exactitud**: ¿qué mide? ¿cómo se detecta y corrige en Python? | *(complete aquí)* |
| 4 | Dimensión de **consistencia**: ¿qué mide? ¿cómo se detecta y corrige en Python? | *(complete aquí)* |
| 5 | Dimensión de **unicidad**: ¿qué mide? ¿cómo se detecta y corrige en Python? | *(complete aquí)* |
| 6 | Dimensión de **oportunidad**: ¿qué mide? ¿cómo afecta a las decisiones? | *(complete aquí)* |
| 7 | ¿Qué es ydata-profiling y qué información proporciona automáticamente? | *(complete aquí)* |
| 8 | ¿Qué es Great Expectations y cómo se usa en pipelines de datos productivos? | *(complete aquí)* |

---
## 🔍 ACTIVIDAD 2: Desarrollo Práctico — Análisis de Calidad de la BD del Proyecto

### 📍 Paso 1: Carga de datos y análisis inicial de calidad

In [ ]:
# Instalar ydata-profiling (ejecutar solo una vez en Colab)
!pip install ydata-profiling -q
print('✅ ydata-profiling instalado')

In [ ]:
# ============================================================
# PASO 1: Carga de datos y análisis inicial
# ============================================================
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# ─── Conectar a la BD del proyecto ───────────────────────────
# Opción A: desde Drive → DB_PATH = '/content/drive/MyDrive/proyecto/proyecto.db'
# Opción B: crear BD de ejemplo en esta sesión
DB_PATH = 'proyecto_calidad.db'
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Crear tablas con datos de ejemplo (INCLUYENDO problemas de calidad intencionales)
cursor.executescript('''
    DROP TABLE IF EXISTS clientes;
    DROP TABLE IF EXISTS productos;
    DROP TABLE IF EXISTS ventas;
    CREATE TABLE clientes (
        id_cliente INTEGER PRIMARY KEY, nombre TEXT, email TEXT,
        ciudad TEXT, edad INTEGER, fecha_reg TEXT
    );
    CREATE TABLE productos (
        id_producto INTEGER PRIMARY KEY, nombre TEXT,
        categoria TEXT, precio REAL, stock INTEGER
    );
    CREATE TABLE ventas (
        id_venta INTEGER PRIMARY KEY, id_cliente INTEGER,
        id_producto INTEGER, cantidad INTEGER,
        fecha_venta TEXT, total REAL
    );
''')

# Datos con problemas de calidad intencionados para el ejercicio
clientes_data = [
    (1,'Ana García','ana@mail.com','Lima',28,'2026-01-10'),
    (2,'Luis Pérez',None,'Arequipa',35,'2026-01-15'),       # email nulo
    (3,'MARIA LOPEZ','maria@mail.com','lima',None,'2026-02-01'), # ciudad inconsistente, edad nula
    (4,'Carlos Ramos','carlos@mail.com','Cusco',200,'2026-02-10'), # edad inválida
    (5,'Ana García','ana@mail.com','Lima',28,'2026-01-10'),  # duplicado exacto
    (6,'Sofia Díaz','sofia@mail.com','Trujillo',22,'2026-03-01'),
    (7,'Pedro Torres',None,'AREQUIPA',45,'2026-03-05'),      # email nulo, ciudad inconsistente
    (8,'Rosa Flores','rosa@mail.com','Lima',-5,'2026-03-10'), # edad negativa
    (9,'José Medina','jose@mail.com','Lima',31,'2026-03-15'),
    (10,'Lucía Vargas','lucia@mail.com','Cusco',29,'2026-03-20'),
]
cursor.executemany('INSERT INTO clientes VALUES (?,?,?,?,?,?)', clientes_data)

productos_data = [
    (1,'Laptop Dell','Electrónica',2499.99,50),
    (2,'Polo Básico','Ropa',None,200),      # precio nulo
    (3,'Arroz 5kg','Alimentos',8.50,500),
    (4,'Auriculares','electronica',149.90,80), # categoría inconsistente
    (5,'Jeans Levi','Ropa',189.00,-10),     # stock negativo
    (6,'Aceite 1L','Alimentos',5.20,300),
    (7,'Laptop Dell','Electrónica',2499.99,50), # duplicado
]
cursor.executemany('INSERT INTO productos VALUES (?,?,?,?,?)', productos_data)

ventas_data = [
    (1,1,1,2,'2026-01-20',4999.98),(2,2,3,5,'2026-01-25',42.50),
    (3,3,2,1,'2026-02-05',None),(4,1,4,1,'2026-02-10',149.90), # total nulo
    (5,6,5,2,'2026-02-15',378.00),(6,9,6,3,'2026-03-01',15.60),
    (7,1,1,0,'2026-03-05',0.00),(8,10,3,10,'2026-03-10',85.00), # cantidad 0
    (9,99,1,1,'2026-03-15',2499.99), # cliente inexistente (FK inválido)
]
cursor.executemany('INSERT INTO ventas VALUES (?,?,?,?,?,?)', ventas_data)
conn.commit()

# Cargar datos
df_cli = pd.read_sql('SELECT * FROM clientes', conn)
df_pro = pd.read_sql('SELECT * FROM productos', conn)
df_ven = pd.read_sql('SELECT * FROM ventas', conn)

print('✅ Datos cargados con problemas de calidad para análisis')
print(f'  Clientes : {len(df_cli)} registros, {df_cli.shape[1]} columnas')
print(f'  Productos: {len(df_pro)} registros, {df_pro.shape[1]} columnas')
print(f'  Ventas   : {len(df_ven)} registros, {df_ven.shape[1]} columnas')

**❓ Pregunta 1:** ¿Cuántos problemas de calidad encontraste en la carga inicial? Clasifícalos por dimensión (completitud, exactitud, consistencia, unicidad, oportunidad).

> **Respuesta:** *(escriba aquí)*

### 📍 Paso 2: Análisis de completitud (valores nulos)

In [ ]:
# ============================================================
# PASO 2: Análisis de completitud
# ============================================================

def analizar_completitud(df, nombre_tabla):
    """Calcula el % de completitud por columna."""
    total = len(df)
    nulos = df.isnull().sum()
    pct_completo = (1 - df.isnull().mean()) * 100
    resultado = pd.DataFrame({
        'columna'      : df.columns,
        'nulos'        : nulos.values,
        'completitud_%': pct_completo.values.round(1),
        'criticidad'   : ['🔴 CRÍTICO' if p < 80 else ('🟡 ALERTA' if p < 95 else '🟢 OK')
                          for p in pct_completo.values]
    })
    score = pct_completo.mean()
    print(f'\n📊 COMPLETITUD — Tabla: {nombre_tabla} | Score: {score:.1f}%')
    print(resultado.to_string(index=False))
    return score

score_cli = analizar_completitud(df_cli, 'clientes')
score_pro = analizar_completitud(df_pro, 'productos')
score_ven = analizar_completitud(df_ven, 'ventas')

print(f'\n📈 SCORE GLOBAL DE COMPLETITUD: {(score_cli + score_pro + score_ven)/3:.1f}%')

# Visualizar
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('🔍 Análisis de Completitud por Tabla', fontsize=13, fontweight='bold')
for ax, (df, nombre) in zip(axes, [(df_cli,'clientes'),(df_pro,'productos'),(df_ven,'ventas')]):
    completitud = (1 - df.isnull().mean()) * 100
    colores = ['#e74c3c' if v < 80 else ('#f39c12' if v < 95 else '#2ecc71') for v in completitud]
    ax.barh(completitud.index, completitud.values, color=colores)
    ax.set_xlim(0, 100)
    ax.axvline(x=95, color='orange', linestyle='--', alpha=0.7, label='Umbral 95%')
    ax.set_title(f'{nombre}\n(Score: {completitud.mean():.1f}%)')
    ax.set_xlabel('Completitud (%)')
plt.tight_layout()
plt.show()

**❓ Pregunta 2:** ¿Cuál es el porcentaje general de completitud de tu BD? ¿Qué tabla presenta más problemas? ¿Qué estrategia de imputación es más adecuada para cada columna con nulos?

> **Respuesta:** *(escriba aquí)*

### 📍 Paso 3: Análisis de unicidad (duplicados) y consistencia

In [ ]:
# ============================================================
# PASO 3: Análisis de unicidad y consistencia
# ============================================================

print('🔄 ANÁLISIS DE UNICIDAD (Duplicados)')
print('='*50)
for df, nombre in [(df_cli,'clientes'),(df_pro,'productos'),(df_ven,'ventas')]:
    dup_total   = df.duplicated().sum()
    dup_pct     = dup_total / len(df) * 100
    estado      = '🔴 CRÍTICO' if dup_pct > 5 else ('🟡 ALERTA' if dup_pct > 1 else '🟢 OK')
    print(f'  {nombre:12s}: {dup_total} duplicados ({dup_pct:.1f}%) → {estado}')
    if dup_total > 0:
        print(f'  Registros duplicados en {nombre}:')
        print(df[df.duplicated(keep=False)].to_string(index=True))

print('\n📐 ANÁLISIS DE CONSISTENCIA')
print('='*50)

# Consistencia: valores fuera de rango / formatos irregulares
problemas_consistencia = []

# Edad inválida (< 0 o > 120)
edad_inv = df_cli[(df_cli['edad'] < 0) | (df_cli['edad'] > 120)].dropna(subset=['edad'])
if len(edad_inv) > 0:
    problemas_consistencia.append(f'Edades inválidas: {len(edad_inv)} registros → {edad_inv["edad"].tolist()}')

# Ciudad con mayúsculas inconsistentes
ciudades_unicas = df_cli['ciudad'].dropna().unique()
ciudades_inconsistentes = [c for c in ciudades_unicas if c != c.title()]
if ciudades_inconsistentes:
    problemas_consistencia.append(f'Ciudades con formato inconsistente: {ciudades_inconsistentes}')

# Stock negativo
stock_neg = df_pro[df_pro['stock'] < 0]
if len(stock_neg) > 0:
    problemas_consistencia.append(f'Stock negativo: {len(stock_neg)} productos → IDs: {stock_neg["id_producto"].tolist()}')

# Cantidad = 0 en ventas
cant_cero = df_ven[df_ven['cantidad'] <= 0]
if len(cant_cero) > 0:
    problemas_consistencia.append(f'Ventas con cantidad ≤ 0: {len(cant_cero)} registros')

# Categorías inconsistentes
cats = df_pro['categoria'].dropna().unique()
cats_inconsistentes = [c for c in cats if c != c.title()]
if cats_inconsistentes:
    problemas_consistencia.append(f'Categorías inconsistentes: {cats_inconsistentes}')

for i, p in enumerate(problemas_consistencia, 1):
    print(f'  {i}. {p}')

print(f'\n  Total problemas de consistencia detectados: {len(problemas_consistencia)}')

**❓ Pregunta 3:** ¿Encontraste duplicados en tu BD? ¿Son duplicados exactos o parciales? ¿Qué estrategia usarías para eliminarlos sin perder información valiosa?

> **Respuesta:** *(escriba aquí)*

### 📍 Paso 4: Pipeline de correcciones automáticas

Implementa las correcciones de calidad y calcula el score antes y después.

In [ ]:
# ============================================================
# PASO 4: Pipeline de correcciones automáticas
# ============================================================

def calcular_score_calidad(df):
    """Calcula un score de calidad global (0-100) para un DataFrame."""
    completitud = (1 - df.isnull().mean()).mean() * 100
    unicidad    = (1 - df.duplicated().mean()) * 100
    return round((completitud * 0.5 + unicidad * 0.5), 1)

# Scores ANTES de correcciones
score_antes = {
    'clientes' : calcular_score_calidad(df_cli),
    'productos': calcular_score_calidad(df_pro),
    'ventas'   : calcular_score_calidad(df_ven)
}

print('📊 SCORES DE CALIDAD ANTES DE CORRECCIONES:')
for tabla, score in score_antes.items():
    print(f'  {tabla:12s}: {score:.1f}/100 {"✅" if score >= 90 else ("⚠️" if score >= 70 else "❌")}')

# ─── PIPELINE DE CORRECCIONES ────────────────────────────────
print('\n🔧 APLICANDO PIPELINE DE CORRECCIONES...')

# 1. Clientes: correcciones
df_cli_limpio = df_cli.copy()
df_cli_limpio = df_cli_limpio.drop_duplicates()                              # Unicidad
df_cli_limpio['ciudad'] = df_cli_limpio['ciudad'].str.title()                # Consistencia: ciudad
df_cli_limpio['email'] = df_cli_limpio['email'].fillna('sin_email@noreply.com') # Completitud
df_cli_limpio.loc[df_cli_limpio['edad'] < 0, 'edad'] = np.nan               # Exactitud: edad negativa
df_cli_limpio.loc[df_cli_limpio['edad'] > 120, 'edad'] = np.nan             # Exactitud: edad imposible
edad_media = df_cli_limpio['edad'].median()
df_cli_limpio['edad'] = df_cli_limpio['edad'].fillna(edad_media)             # Imputación: mediana

# 2. Productos: correcciones
df_pro_limpio = df_pro.copy()
df_pro_limpio = df_pro_limpio.drop_duplicates(subset=['nombre','categoria'])
df_pro_limpio['categoria'] = df_pro_limpio['categoria'].str.title()          # Consistencia
precio_mediana = df_pro_limpio['precio'].median()
df_pro_limpio['precio'] = df_pro_limpio['precio'].fillna(precio_mediana)     # Completitud
df_pro_limpio.loc[df_pro_limpio['stock'] < 0, 'stock'] = 0                  # Exactitud

# 3. Ventas: correcciones
df_ven_limpio = df_ven.copy()
ids_validos = df_cli_limpio['id_cliente'].tolist()
df_ven_limpio = df_ven_limpio[df_ven_limpio['id_cliente'].isin(ids_validos)] # Consistencia referencial
df_ven_limpio = df_ven_limpio[df_ven_limpio['cantidad'] > 0]                 # Exactitud
df_ven_limpio['total'] = df_ven_limpio['total'].fillna(
    df_ven_limpio['cantidad'] * 0  # Placeholder: requiere JOIN con precios
)                                                                             # Completitud

# Scores DESPUÉS de correcciones
score_despues = {
    'clientes' : calcular_score_calidad(df_cli_limpio),
    'productos': calcular_score_calidad(df_pro_limpio),
    'ventas'   : calcular_score_calidad(df_ven_limpio)
}

print('\n📊 SCORES DE CALIDAD DESPUÉS DE CORRECCIONES:')
for tabla in score_antes:
    antes   = score_antes[tabla]
    despues = score_despues[tabla]
    mejora  = despues - antes
    print(f'  {tabla:12s}: {antes:.1f} → {despues:.1f} | Mejora: +{mejora:.1f} pts')

print(f'\n📉 Registros eliminados:')
print(f'  Clientes : {len(df_cli)} → {len(df_cli_limpio)} ({len(df_cli)-len(df_cli_limpio)} eliminados)')
print(f'  Productos: {len(df_pro)} → {len(df_pro_limpio)} ({len(df_pro)-len(df_pro_limpio)} eliminados)')
print(f'  Ventas   : {len(df_ven)} → {len(df_ven_limpio)} ({len(df_ven)-len(df_ven_limpio)} eliminados)')

In [ ]:
# Gráfico: Comparación de scores antes/después
tablas_nombres = list(score_antes.keys())
scores_a = list(score_antes.values())
scores_d = list(score_despues.values())

x = np.arange(len(tablas_nombres))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, scores_a, width, label='Antes', color='#e74c3c', alpha=0.8)
bars2 = ax.bar(x + width/2, scores_d, width, label='Después', color='#2ecc71', alpha=0.8)

ax.set_ylabel('Score de Calidad (0-100)')
ax.set_title('📊 Mejora de Calidad — Antes vs Después del Pipeline', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([t.upper() for t in tablas_nombres])
ax.set_ylim(0, 105)
ax.axhline(y=90, color='orange', linestyle='--', alpha=0.7, label='Umbral mínimo (90)')
ax.legend()

for bar in bars1 + bars2:
    ax.annotate(f'{bar.get_height():.1f}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('mejora_calidad_pipeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico de mejora guardado')

**❓ Pregunta 4:** Calcula el score de calidad antes y después. ¿En cuánto mejoró el score? ¿Qué dimensión tuvo la mayor mejora?

> **Respuesta:** *(escriba aquí)*

### 📍 Paso 5: Reporte automatizado con ydata-profiling

In [ ]:
# ============================================================
# PASO 5: Reporte automatizado con ydata-profiling
# ============================================================

try:
    from ydata_profiling import ProfileReport

    # Reporte de la tabla principal (clientes limpia)
    profile = ProfileReport(
        df_cli_limpio,
        title='Reporte de Calidad — Clientes (Limpio) | LAB D13',
        minimal=True,   # minimal=True para Colab (más rápido)
        explorative=True
    )
    profile.to_file('reporte_calidad_clientes.html')
    print('✅ Reporte ydata-profiling generado: reporte_calidad_clientes.html')
    print('   → Abre el archivo HTML para ver el reporte interactivo completo')
    print('\n📊 RESUMEN DEL REPORTE:')
    print(f'  Registros   : {len(df_cli_limpio)}')
    print(f'  Variables   : {df_cli_limpio.shape[1]}')
    print(f'  Nulos totales: {df_cli_limpio.isnull().sum().sum()}')
    print(f'  Duplicados  : {df_cli_limpio.duplicated().sum()}')

except ImportError:
    print('⚠️  ydata-profiling no disponible. Generando resumen básico...')
    print(df_cli_limpio.describe(include='all').to_string())

**❓ Pregunta 5:** ¿Qué alertas detectó ydata-profiling en tu dataset? ¿Hay correlaciones altas o distribuciones sesgadas? ¿Qué implica esto para el análisis?

> **Respuesta:** *(escriba aquí)*

### 📍 Paso 6: Exportación de datos limpios a SQLite

In [ ]:
# ============================================================
# PASO 6: Guardar datos limpios en SQLite
# ============================================================

# Guardar tablas limpias en SQLite
df_cli_limpio.to_sql('clientes_limpio', conn, if_exists='replace', index=False)
df_pro_limpio.to_sql('productos_limpio', conn, if_exists='replace', index=False)
df_ven_limpio.to_sql('ventas_limpio', conn, if_exists='replace', index=False)
conn.commit()

print('✅ DATOS LIMPIOS GUARDADOS EN SQLITE')
print('='*50)

# Verificar
tablas_bd = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
for tabla in tablas_bd['name']:
    count = pd.read_sql(f'SELECT COUNT(*) as n FROM {tabla}', conn).iloc[0,0]
    print(f'  {tabla:22s}: {count:4} registros')

# Exportar también como CSV
df_cli_limpio.to_csv('clientes_limpio.csv', index=False, encoding='utf-8-sig')
df_pro_limpio.to_csv('productos_limpio.csv', index=False, encoding='utf-8-sig')
df_ven_limpio.to_csv('ventas_limpio.csv', index=False, encoding='utf-8-sig')
print('\n✅ CSVs exportados: clientes_limpio.csv, productos_limpio.csv, ventas_limpio.csv')

conn.close()
print('\n🎉 Pipeline de calidad completado exitosamente')

**❓ Pregunta 6:** ¿Cuántos registros perdiste durante el proceso de limpieza? ¿Fue aceptable esa pérdida? ¿Qué estrategias alternativas hubieran reducido la pérdida de datos?

> **Respuesta:** *(escriba aquí)*

---
## 🏢 ACTIVIDAD 3: Caso de Estudio — Reporte de Calidad Final del Proyecto

**Contexto:** Tu equipo está a punto de entregar el proyecto final. El director de datos solicita un reporte de calidad que demuestre que los datos son confiables y están listos para análisis. Debes identificar problemas pendientes y proponer un plan de mejora.

### ❓ Pregunta A — Scorecard de Calidad
Para cada dimensión asigna un puntaje del 1 al 5 y justifica con evidencia del análisis.

> **Respuesta:**
>
> | Dimensión | Puntaje (1-5) | Problemas encontrados | Acciones correctivas |
> |-----------|:-------------:|----------------------|---------------------|
> | Completitud | *(N)* | *(completar)* | *(completar)* |
> | Exactitud | *(N)* | *(completar)* | *(completar)* |
> | Consistencia | *(N)* | *(completar)* | *(completar)* |
> | Unicidad | *(N)* | *(completar)* | *(completar)* |
> | Oportunidad | *(N)* | *(completar)* | *(completar)* |
> | **SCORE GLOBAL** | ***(promedio)*** | | |

### ❓ Pregunta B — 3 Problemas Críticos
Identifica los 3 problemas de calidad más críticos. Para cada uno: describe el problema, cuantifica el impacto y propón la corrección.

> **Respuesta:**
>
> **Problema 1:**
> - Descripción: *(completar)*
> - % registros afectados: *(completar)*
> - Corrección implementada: *(completar)*
>
> **Problema 2:**
> - Descripción: *(completar)*
> - % registros afectados: *(completar)*
> - Corrección implementada: *(completar)*
>
> **Problema 3:**
> - Descripción: *(completar)*
> - % registros afectados: *(completar)*
> - Corrección implementada: *(completar)*

### ❓ Pregunta C — Diseño de Validaciones con Great Expectations
Diseña al menos 5 expectativas (expectations) para tu BD usando la sintaxis de Great Expectations.

> **Respuesta:**
>
> ```python
> # Ejemplo de expectations para tu BD
> # (Reemplaza con columnas reales de tu proyecto)
> 
> # Expectativa 1: email no puede ser nulo
> # expect_column_values_to_not_be_null(column='email')
> 
> # Expectativa 2: precio debe ser mayor a 0
> # expect_column_values_to_be_between(column='precio', min_value=0)
> 
> # Expectativa 3: ciudad debe ser un valor de una lista
> # expect_column_values_to_be_in_set(column='ciudad', value_set=['Lima','Arequipa','Cusco'])
> 
> # Expectativa 4: *(completar)*
> # expect_...(column='...', ...)
> 
> # Expectativa 5: *(completar)*
> # expect_...(column='...', ...)
> ```

### ❓ Pregunta D — Impacto en el Modelo ML
¿Cómo afectan los problemas de calidad encontrados a las conclusiones del modelo ML desarrollado en las semanas anteriores?

> **Respuesta:**
>
> **Impacto 1:**
> - Problema: *(completar)*
> - Efecto en el modelo: *(completar)*
> - Mitigación aplicada: *(completar)*
>
> **Impacto 2:**
> - Problema: *(completar)*
> - Efecto en el modelo: *(completar)*
> - Mitigación aplicada: *(completar)*

---
## ✅ Conclusiones

1. ___________________________________________________________________

2. ___________________________________________________________________

3. ___________________________________________________________________

---
## 📖 Material Complementario

1. DAMA International (2017). *DAMA-DMBOK*. Technics Publications. Capítulo 13: Data Quality.
2. Redman, T. C. (2001). *Data Quality: The Field Guide*. Digital Press. Capítulos 2-4.
3. ydata-profiling Documentation: https://docs.profiling.ydata.ai/latest/
4. Great Expectations Documentation: https://docs.greatexpectations.io/docs/

---
*© 2026 — Fundamentos de Gestión de Datos — TECSUP | Docente: Pilar Rocío Sayán Mejía*